# Anomaly Detection for Network & System Logs

This notebook demonstrates the anomaly detection pipeline in `scripts/anomaly_detection/`:
- Feature extraction from raw log strings
- Isolation Forest detector
- Local Outlier Factor detector
- High-level `AnomalyDetector` class

In [ ]:
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.anomaly_detection.network_anomaly_detector import (
    extract_log_features, AnomalyDetector, IsolationForestDetector, LOFDetector
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Simulate Log Data

In [ ]:
normal_logs = [
    'Jan  1 00:01:15 server01 sshd[1234]: INFO Accepted password for user from 192.168.1.1',
    'Jan  1 00:02:30 server01 cron[5678]: INFO (root) CMD (/usr/bin/backup.sh)',
    'Jan  1 00:03:45 server01 kernel: INFO eth0: Link is up at 1 Gbps',
    'Jan  1 00:04:00 server01 systemd[1]: INFO Started daily cleanup service',
    'Jan  1 00:05:12 server01 sshd[1234]: INFO session opened for user deploy',
] * 40

anomaly_logs = [
    'CRITICAL server01 sshd[9999]: ERROR Failed password for invalid user admin from 10.0.0.1 – BRUTE FORCE DETECTED',
    'FATAL kernel: ERROR segfault at 0x0000deadbeef exception traceback timeout refused',
    'ERROR webserver: CRITICAL upstream timed out (110: Connection timed out) denied failed',
] * 5

df_logs = pd.DataFrame({'log': normal_logs + anomaly_logs})
df_logs = df_logs.sample(frac=1, random_state=42).reset_index(drop=True)
print('Dataset shape:', df_logs.shape)

## 2. Log Feature Extraction

In [ ]:
features = extract_log_features(df_logs['log'])
print(features.describe())
features.head()

## 3. Isolation Forest Detection

In [ ]:
detector = AnomalyDetector(method='isolation_forest', contamination=0.05)
results = detector.fit_detect(df_logs)
print(f"Detected anomalies: {results['is_anomaly'].sum()} / {len(results)}")
results[results['is_anomaly']]['log'].head(10)

## 4. Anomaly Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(
    results.index,
    results['anomaly_score'],
    c=results['is_anomaly'].map({True: 'red', False: 'steelblue'}),
    alpha=0.6
)
ax.axhline(0, color='gray', linestyle='--', lw=0.8)
ax.set_xlabel('Log index')
ax.set_ylabel('Anomaly score (lower = more anomalous)')
ax.set_title('Anomaly Scores per Log Entry')
plt.tight_layout()
plt.show()

## 5. Local Outlier Factor Detection

In [ ]:
lof_detector = AnomalyDetector(method='lof', contamination=0.05)
lof_results = lof_detector.fit_detect(df_logs)
print(f"LOF detected anomalies: {lof_results['is_anomaly'].sum()} / {len(lof_results)}")